In [53]:
import json 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re
import random
import os
rng = random.Random(42)

Generate and save data for Coherence model comparison.

In [2]:
def choose_device(device):
    if device==0:
        return 'cuda:0'
    elif device==1:
        return 'cuda:1'
    elif device ==-1:
        return 'cpu'
    else:
        raise Exception('Return 0 or 1 for GPUs or -1 for CPU')

def load_device(cuda_id):
    cuda = choose_device(cuda_id)
    device = torch.device(cuda if torch.cuda.is_available() else "cpu")
    return device

def load_AutoModel(model_id,cuda_id):
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side = "left") 
    tokenizer.pad_token_id = tokenizer.eos_token_id #required in llama because no padding token is defined
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]
    device = load_device(cuda_id)
    model = model.to(device)

    return tokenizer, model, device, terminators

def load_prompts(prompt_type, prompt_topic):
    #load prompt from .txt
    if prompt_type == 'completion':
        with open('../prompts-comp/'+prompt_topic+'.txt') as file:
            prompt = file.read()
    elif prompt_type == 'generation':
        with open('../prompts-gen/'+prompt_topic+'.txt') as file:
            prompt = file.read()
        prompt = json.loads(prompt, strict=False)
    return prompt

def prepare_llama_prompt(tokenizer, prompt, device):
    text = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False) 
    inputs = tokenizer(text, padding="longest", return_tensors="pt") #transform into pt tensors
    inputs = {key: val.to(device) for key, val in inputs.items()} #move inputs into cuda
    return inputs

def llama_gen(model, inputs, tokenizer, terminators, num_generations):
    generations = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=terminators,
        num_return_sequences=num_generations  
    )

    return generations

## Prompt-based data

#### Generate the three prompts, saved them.

In [19]:
def load_prompt_and_gen(prompt_type, prompt_topic, model, tokenizer, terminators, device):
    prompt = load_prompts(prompt_type, prompt_topic)
    model_inputs = prepare_llama_prompt(tokenizer, prompt, device)
    gen_ids = llama_gen(model, model_inputs, tokenizer, terminators, num_generations=100)

    return gen_ids, model_inputs

def save_list_to_json(file_path, file_name, list_to_save):
    final_path = file_path + file_name + ".json"
    with open(final_path, "w", encoding="utf-8") as f:
        json.dump(list_to_save, f, ensure_ascii=False, indent=2)

In [4]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer, model, device, terminators = load_AutoModel(model_id, cuda_id=0)

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 134.06it/s]


In [5]:
viktor_ids, viktor_inputs = load_prompt_and_gen(prompt_type="generation", prompt_topic="viktor", model=model, tokenizer=tokenizer, terminators=terminators, device=device)

In [11]:
viktor_stories_list = tokenizer.batch_decode(viktor_ids[:,viktor_inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [6]:
prague_ids, prague_inputs = load_prompt_and_gen(prompt_type="generation", prompt_topic="prague", model=model, tokenizer=tokenizer, terminators=terminators, device=device)

In [12]:
prague_stories_list = tokenizer.batch_decode(prague_ids[:,prague_inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [7]:
sciencefic_ids, sciencefic_inputs = load_prompt_and_gen(prompt_type="generation", prompt_topic="sciencefic", model=model, tokenizer=tokenizer, terminators=terminators, device=device)

In [13]:
sciencefic_stories_list = tokenizer.batch_decode(sciencefic_ids[:,sciencefic_inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
saving_path = "../model-comparison/data/clean/"

save_list_to_json(file_path=saving_path, file_name="viktor_100_narratives", list_to_save=viktor_stories_list)
save_list_to_json(file_path=saving_path, file_name="prague_100_narratives", list_to_save=prague_stories_list)
save_list_to_json(file_path=saving_path, file_name="sciencefic_100_narratives", list_to_save=sciencefic_stories_list)

#### Shuffle tokens, sentences and sentences and save them.

Shuffle tokens

In [23]:
def get_special_ids_set(tokenizer):
    #ensure pad_token_id at least exists
    if getattr(tokenizer, "pad_token_id", None) is None:
        raise ValueError("tokenizer.pad_token_id is not set. Set it or use a tokenizer with pad token.")
    
    return set(tokenizer.all_special_ids)

def strip_special_tokens(tensor_ids, tokenizer):
    clean_sequences = []
    lengths = []
    N, L = tensor_ids.shape
    special_ids = get_special_ids_set(tokenizer)

    for i in range(N):
        row = tensor_ids[i]
        ids = row.tolist()

        first, last = None, None
        for idx, tok in enumerate(ids): #find first non special token
            if tok not in special_ids:
                first = idx #save it
                break
    
        for idx in range(len(ids)-1, -1, -1): #iterate from the back of the tensor 
            if ids[idx] not in special_ids: #until a normal token
                last = idx #save it
                break
            
        seq = row[first:last+1].clone()
        clean_sequences.append(seq)
        lengths.append(seq.shape[0])

    return clean_sequences, lengths

def shuffle_tensor_list(tensor_list):

    shuffled_list = []
    for t in tensor_list:
        flat = t.flatten()
        perm = torch.randperm(flat.size(0), device=t.device)
        shuffled = flat[perm].view_as(t)
        shuffled_list.append(shuffled)
    return shuffled_list

In [29]:
viktor_tensor_prompt_size = viktor_inputs["input_ids"].shape[1]
viktor_gen_only = viktor_ids[:,viktor_tensor_prompt_size:]
viktor_clean,_ = strip_special_tokens(viktor_gen_only,tokenizer)
shuffled_tokens_viktor = shuffle_tensor_list(viktor_clean)
shuffled_tokens_viktor_text = tokenizer.batch_decode(shuffled_tokens_viktor)

In [30]:
prague_tensor_prompt_size = prague_inputs["input_ids"].shape[1]
prague_gen_only = prague_ids[:,prague_tensor_prompt_size:]
prague_clean,_ = strip_special_tokens(prague_gen_only,tokenizer)
shuffled_tokens_prague = shuffle_tensor_list(prague_clean)
shuffled_tokens_prague_text = tokenizer.batch_decode(shuffled_tokens_prague)

In [31]:
sciencefic_tensor_prompt_size = sciencefic_inputs["input_ids"].shape[1]
sciencefic_gen_only = sciencefic_ids[:,sciencefic_tensor_prompt_size:]
sciencefic_clean,_ = strip_special_tokens(sciencefic_gen_only,tokenizer)
shuffled_tokens_sciencefic = shuffle_tensor_list(sciencefic_clean)
shuffled_tokens_sciencefic_text = tokenizer.batch_decode(shuffled_tokens_sciencefic)

In [36]:
shuffled_tokens_path = "../model-comparison/data/shuffled-tokens/"

save_list_to_json(file_path=shuffled_tokens_path, file_name="viktor_shuffled_tokens", list_to_save=shuffled_tokens_viktor_text)
save_list_to_json(file_path=shuffled_tokens_path, file_name="prague_shuffled_tokens", list_to_save=shuffled_tokens_prague_text)
save_list_to_json(file_path=shuffled_tokens_path, file_name="sciencefic_shuffled_tokens", list_to_save=shuffled_tokens_sciencefic_text)

Shuffled sentences

In [ ]:
def split_sentences(text):
    """
    Split text into word tokens, preserving punctuation marks as separate tokens.
    Example: "Hello, world!" -> ["Hello", ",", "world", "!"]
    """
    text = text.strip()
    if not text:
        return []
    # Split on word boundaries while keeping punctuation
    tokens = re.findall(r"\w+|[^\w\s]", text, re.UNICODE)
    return tokens

def shuffle_sentences(text, rng=None):
    """
    Shuffle all sentences in a text, preserving punctuation as tokens.
    The RNG argument is optional; if provided, must support .shuffle().
    Ensures that the shuffled output is actually different from the original.
    """
    tokens = split_sentences(text)
    if len(tokens) <= 1:
        return text

    shuffled = tokens[:]
    if rng is not None:
        rng.shuffle(shuffled)
    else:
        random.shuffle(shuffled)

    # Ensure it's actually different
    if shuffled == tokens:
        shuffled = tokens[1:] + tokens[:1]

    # Join with spaces, but remove spaces before punctuation
    shuffled_text = " ".join(shuffled)
    shuffled_text = re.sub(r'\s+([?.!,;:])', r'\1', shuffled_text)
    return shuffled_text

In [ ]:
viktor_shuffled_sentences = [shuffle_sentences(s,rng) for s in viktor_stories_list]
prague_shuffled_sentences = [shuffle_sentences(s,rng) for s in prague_stories_list]
sciencefic_shuffled_sentences = [shuffle_sentences(s,rng) for s in sciencefic_stories_list]

In [ ]:
shuffled_sentences_path = "../model-comparison/data/shuffled-sentences/"

save_list_to_json(file_path=shuffled_sentences_path, file_name="viktor_shuffled_sentences", list_to_save=viktor_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="prague_shuffled_sentences", list_to_save=prague_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="sciencefic_shuffled_sentences", list_to_save=sciencefic_shuffled_sentences)

Shuffled sentences

In [47]:
def split_sentences(text):
    """
    Fast splitter on . ! ?
    """
    text = text.strip()
    if not text:
        return []
    sents = re.split(r'(?<=[.!?])\s+', text)
    return [s for s in sents if s]

def shuffle_sentences(text, rng):
    sents = split_sentences(text)
    if len(sents) <= 1:
        return text
    
    shuffled = sents[:] #copy
    #rng.shuffle(shuffled)
    random.shuffle(shuffled)

    #ensure itss actually different 
    if shuffled == sents:
        shuffled = sents[1:] + sents[:1]
    return " ".join(shuffled) #join sentences again

In [48]:
viktor_shuffled_sentences = [shuffle_sentences(s,rng) for s in viktor_stories_list]
prague_shuffled_sentences = [shuffle_sentences(s,rng) for s in prague_stories_list]
sciencefic_shuffled_sentences = [shuffle_sentences(s,rng) for s in sciencefic_stories_list]

In [52]:
shuffled_sentences_path = "../model-comparison/data/shuffled-sentences/"

save_list_to_json(file_path=shuffled_sentences_path, file_name="viktor_shuffled_sentences", list_to_save=viktor_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="prague_shuffled_sentences", list_to_save=prague_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="sciencefic_shuffled_sentences", list_to_save=sciencefic_shuffled_sentences)

## Paraphrased-based data

Load - remove clause numbers.

In [58]:
def remove_clause_numbers(story: str) -> str:
    #d+ - one or more digits, \. - dot, \s* - newlines and spaces
    cleaned = re.sub(r"\d+\.\s*", "", story) 
    #remove additional spaces
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

In [55]:
dir_path = "../rephrasings"

gpt4_stories = {}
gpt5_stories = {}
for i in range(1, 4):
    file_base = f"story{i}.json"
    path_base = os.path.join(dir_path,file_base)
    with open(path_base, "r", encoding="utf-8") as f:
        gpt4_stories[i] = json.load(f)
    
    file_costly = f"{i}_high_temp_costly.json"
    path_costly = os.path.join(dir_path,file_costly)
    with open(path_costly, "r", encoding="utf-8") as f:
        gpt5_stories[i] = json.load(f)

In [59]:
gpt4_cleaned_1 = [remove_clause_numbers(story) for story in gpt4_stories[1]]
gpt4_cleaned_2 = [remove_clause_numbers(story) for story in gpt4_stories[2]]
gpt4_cleaned_3 = [remove_clause_numbers(story) for story in gpt4_stories[3]]

gpt5_cleaned_1 = [remove_clause_numbers(story) for story in gpt5_stories[1]]
gpt5_cleaned_2 = [remove_clause_numbers(story) for story in gpt5_stories[2]]
gpt5_cleaned_3 = [remove_clause_numbers(story) for story in gpt5_stories[3]]

In [61]:
clean_path = "../model-comparison/data/clean/"

save_list_to_json(file_path=clean_path, file_name="gpt4_para1", list_to_save=gpt4_cleaned_1)
save_list_to_json(file_path=clean_path, file_name="gpt4_para2", list_to_save=gpt4_cleaned_2)
save_list_to_json(file_path=clean_path, file_name="gpt4_para3", list_to_save=gpt4_cleaned_3)
save_list_to_json(file_path=clean_path, file_name="gpt5_para1", list_to_save=gpt5_cleaned_1)
save_list_to_json(file_path=clean_path, file_name="gpt5_para2", list_to_save=gpt5_cleaned_2)
save_list_to_json(file_path=clean_path, file_name="gpt5_para3", list_to_save=gpt5_cleaned_3)

#### Shuffled tokens, sentences and sentences and save them.

In [62]:
def tokenize_to_infer(text,tokenizer):
    tokens = tokenizer(text, padding="longest", return_tensors="pt") #transform into pt tensors
    return {key: val.to(device) for key, val in tokens.items()} #move inputs into cuda.

Shuffled tokens

In [ ]:
def from_paraphrases_to_shuffled_tokens(cleaned_paraphrases_list, tokenizer):
    tokenized_list = tokenize_to_infer(cleaned_paraphrases_list, tokenizer)
    tokenized_without_special_tokens,_ = strip_special_tokens(tensor_ids=tokenized_list["input_ids"],tokenizer=tokenizer)
    shuffled_tokens = shuffle_tensor_list(tensor_list=tokenized_without_special_tokens)

    return tokenizer.batch_decode(shuffled_tokens)

In [82]:
gpt4_para1_tokens_shuffled = from_paraphrases_to_shuffled_tokens(gpt4_cleaned_1, tokenizer)
gpt4_para2_tokens_shuffled = from_paraphrases_to_shuffled_tokens(gpt4_cleaned_2, tokenizer)
gpt4_para3_tokens_shuffled = from_paraphrases_to_shuffled_tokens(gpt4_cleaned_3, tokenizer)

gpt5_para1_tokens_shuffled = from_paraphrases_to_shuffled_tokens(gpt5_cleaned_1, tokenizer)
gpt5_para2_tokens_shuffled = from_paraphrases_to_shuffled_tokens(gpt5_cleaned_2, tokenizer)
gpt5_para3_tokens_shuffled = from_paraphrases_to_shuffled_tokens(gpt5_cleaned_3, tokenizer)

In [83]:
save_list_to_json(file_path=shuffled_tokens_path, file_name="gpt4_para1_tokens_shuffled", list_to_save=gpt4_para1_tokens_shuffled)
save_list_to_json(file_path=shuffled_tokens_path, file_name="gpt4_para2_tokens_shuffled", list_to_save=gpt4_para2_tokens_shuffled)
save_list_to_json(file_path=shuffled_tokens_path, file_name="gpt4_para3_tokens_shuffled", list_to_save=gpt4_para3_tokens_shuffled)
save_list_to_json(file_path=shuffled_tokens_path, file_name="gpt5_para1_tokens_shuffled", list_to_save=gpt5_para1_tokens_shuffled)
save_list_to_json(file_path=shuffled_tokens_path, file_name="gpt5_para2_tokens_shuffled", list_to_save=gpt5_para2_tokens_shuffled)
save_list_to_json(file_path=shuffled_tokens_path, file_name="gpt5_para3_tokens_shuffled", list_to_save=gpt5_para3_tokens_shuffled)

Shuffled sentences

In [ ]:
gpt4_para1_shuffled_sentences = [shuffle_sentences(story) for story in gpt4_cleaned_1]
gpt4_para2_shuffled_sentences = [shuffle_sentences(story) for story in gpt4_cleaned_2]
gpt4_para3_shuffled_sentences = [shuffle_sentences(story) for story in gpt4_cleaned_3]

In [ ]:
gpt5_para1_shuffled_sentences = [shuffle_sentences(story) for story in gpt5_cleaned_1]
gpt5_para2_shuffled_sentences = [shuffle_sentences(story) for story in gpt5_cleaned_2]
gpt5_para3_shuffled_sentences = [shuffle_sentences(story) for story in gpt5_cleaned_3]

In [91]:
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt4_para1_sentences_shuffled", list_to_save=gpt4_para1_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt4_para2_sentences_shuffled", list_to_save=gpt4_para2_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt4_para3_sentences_shuffled", list_to_save=gpt4_para3_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt5_para1_sentences_shuffled", list_to_save=gpt5_para1_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt5_para2_sentences_shuffled", list_to_save=gpt5_para2_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt5_para3_sentences_shuffled", list_to_save=gpt5_para3_shuffled_sentences)

Shuffled sentences

In [92]:
def split_sentences(story):
    #split on ., ?, or ! followed by space
    sentences = re.split(r'(?<=[.!?])\s+', story)
    return [s for s in sentences if s]  # remove empty strings

def shuffle_sentences(list_of_sentences):
    random.shuffle(list_of_sentences)
    return " ".join(list_of_sentences)

def shuffle_story(story):
    return shuffle_sentences(split_sentences(story))

In [94]:
gpt4_para1_shuffled_sentences = [shuffle_story(story) for story in gpt4_cleaned_1]
gpt4_para2_shuffled_sentences = [shuffle_story(story) for story in gpt4_cleaned_2]
gpt4_para3_shuffled_sentences = [shuffle_story(story) for story in gpt4_cleaned_3]

In [95]:
gpt5_para1_shuffled_sentences = [shuffle_story(story) for story in gpt5_cleaned_1]
gpt5_para2_shuffled_sentences = [shuffle_story(story) for story in gpt5_cleaned_2]
gpt5_para3_shuffled_sentences = [shuffle_story(story) for story in gpt5_cleaned_3]

In [97]:
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt4_para1_sentences_shuffled", list_to_save=gpt4_para1_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt4_para2_sentences_shuffled", list_to_save=gpt4_para2_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt4_para3_sentences_shuffled", list_to_save=gpt4_para3_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt5_para1_sentences_shuffled", list_to_save=gpt5_para1_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt5_para2_sentences_shuffled", list_to_save=gpt5_para2_shuffled_sentences)
save_list_to_json(file_path=shuffled_sentences_path, file_name="gpt5_para3_sentences_shuffled", list_to_save=gpt5_para3_shuffled_sentences)